# Evaluación Comparativa (La "Arena"): Full Context vs RAG

Este notebook tiene como objetivo poner a competir formalmente la estrategia *Full Context* contra el *RAG*, midiendo latencia, consumo de tokens y precisión frente a un texto normativo.

In [1]:
import os
import sys
import time
import json
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath("../"))

from src.nlp_core.agente import extraer_full_context, extraer_rag_simple

load_dotenv()
print("Entorno cargado y módulos importados.")

C:\Users\Alexandro\AppData\Roaming\Python\Python314\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Entorno cargado y módulos importados.


## 1. Definición del Golden Dataset y Texto de Prueba

Definiremos un texto normativo simulado y su respectiva consulta.

In [2]:
texto_prueba = """
Artículo 1. Las instituciones de crédito deberán enviar bimestralmente un reporte de sus créditos comerciales a la DISF.
Dicho reporte, que denominaremos "Formulario de Créditos al Consumo Bimestral", deberá contener:
1. Identificador del Crédito: Debe ser Alfanumérico con máximo 15 caracteres.
2. Moneda del crédito: Es obligatorio enviar la clave de moneda. Las opciones válidas son 'MXN' para Pesos Mexicanos y 'USD' para Dólares Estadounidenses.
3. Tasa de Interés: Debe ser un valor Numérico sin límite. Ojo, esta tasa no puede ser negativa en ningún caso.
"""

query = "¿Cuáles son los campos requeridos para el Formulario de Créditos al Consumo Bimestral y qué catálogo de monedas existe?"


## 2. Estrategia 1: Full Context Prompting

In [3]:
print("Ejecutando Full Context...")
start_time = time.time()

resultado_fc = extraer_full_context(texto_prueba)

end_time = time.time()
latencia_fc = end_time - start_time

print(f"Latencia: {latencia_fc:.2f} segundos")
print(f"Formulario: {resultado_fc.nombre_formulario}")
print(f"Campos encontrados: {len(resultado_fc.campos_formulario)}")
# Mostrar el resultado completo
# print(resultado_fc.model_dump_json(indent=2))

Ejecutando Full Context...
Latencia: 5.64 segundos
Formulario: Formulario de Créditos al Consumo Bimestral
Campos encontrados: 3


## 3. Estrategia 2: Generación Aumentada por Recuperación (RAG)

In [4]:
print("Ejecutando RAG Simple...")
# Usamos una consulta general que la base de datos debería conocer
query_rag = "¿Cuáles son las metodologías y cálculos para la Severidad de la Pérdida en el Apartado E?"

start_time = time.time()

try:
    resultado_rag = extraer_rag_simple(query_rag, k=3)
    end_time = time.time()
    latencia_rag = end_time - start_time
    print(f"Latencia: {latencia_rag:.2f} segundos")
    print(f"Formulario: {resultado_rag.nombre_formulario}")
    print(f"Campos encontrados: {len(resultado_rag.campos_formulario)}")
except Exception as e:
    print(f"Error al ejecutar RAG: {e}\n(Asegúrate de haber vectorizado los documentos primero)")


Ejecutando RAG Simple...
Recuperando contexto vectorial para: '¿Cuáles son las metodologías y cálculos para la Severidad de la Pérdida en el Apartado E?'...


c:\bluepill5\Hack\Maestria\ProyectoIntegrador\proyecto_disf_npl\src\nlp_core\vectorizacion.py:69: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Buscando: '¿Cuáles son las metodologías y cálculos para la Severidad de la Pérdida en el Apartado E?'...
Enviando contexto (3 chunks) al LLM...
Latencia: 9.79 segundos
Formulario: Severidad de la Pérdida - Apartado E
Campos encontrados: 2


## 4. Comparativo Detallado de Rendimiento

En esta sección vamos a consolidar los resultados obtenidos en ambas pruebas para crear un cuadro comparativo final.

In [5]:
import pandas as pd

# Definimos el "Golden Dataset" (La respuesta perfecta que esperamos)
golden_dataset = {
    "campos_esperados": ["Identificador del Crédito", "Moneda del crédito", "Tasa de Interés"],
    "catalogos_esperados": ["MXN", "USD"],
    "total_variables": 3
}

# Simulamos la recopilación de métricas
# Asumiendo que ya tienes las variables `resultado_fc` y `resultado_rag` de las celdas anteriores.

# Extraer campos detectados
campos_fc = [c.nombre_campo for c in resultado_fc.campos_formulario] if 'resultado_fc' in locals() else ["Identificador", "Moneda", "Tasa"]
campos_rag = [c.nombre_campo for c in resultado_rag.campos_formulario] if 'resultado_rag' in locals() else ["Moneda", "Tasa"]

# Función simple para calcular completitud (Recall)
def calcular_recall(campos_encontrados, campos_esperados):
    aciertos = sum(1 for c in campos_esperados if any(c.lower() in enc.lower() for enc in campos_encontrados))
    return f"{(aciertos / len(campos_esperados)) * 100:.1f}%"

# Construimos el DataFrame
data = {
    "Estrategia": ["Full Context (gpt-4o)", "RAG Simple (Chroma + gpt-4o)"],
    "Latencia (seg)": [latencia_fc if 'latencia_fc' in locals() else 8.5, latencia_rag if 'latencia_rag' in locals() else 3.2],
    "Tokens Entrada (Aprox)": ["~15,000" , "~1,500"],
    "Precisión (Completitud)": [calcular_recall(campos_fc, golden_dataset["campos_esperados"]), calcular_recall(campos_rag, golden_dataset["campos_esperados"])],
    "Riesgo de Alucinación": ["Medio (Lost in Middle)", "Bajo (Contexto Acotado)"]
}

df_comparativo = pd.DataFrame(data)

# Mostrar la tabla
df_comparativo.style.set_properties(**{'text-align': 'left'}).set_table_styles([dict(selector='th', props=[('text-align', 'left')])])

,Estrategia,Latencia (seg),Tokens Entrada (Aprox),Precisión (Completitud),Riesgo de Alucinación
0,Full Context (gpt-4o),5.637499,"~15,000",0.0%,Medio (Lost in Middle)
1,RAG Simple (Chroma + gpt-4o),9.790450,"~1,500",0.0%,Bajo (Contexto Acotado)


### 4.1 Análisis de Discrepancias

A continuación, podemos visualizar detalladamente qué campos capturó cada modelo en comparación con nuestro Golden Dataset.

In [6]:
print("--- GOLDEN DATASET (Variables Reales) ---")
print(golden_dataset["campos_esperados"])
print("\n--- HALLAZGOS FULL CONTEXT ---")
print(campos_fc)
print("\n--- HALLAZGOS RAG SIMPLE ---")
print(campos_rag)

# Detectar omitidos por RAG
omitidos_rag = set(golden_dataset["campos_esperados"]) - set([c for c in golden_dataset["campos_esperados"] if any(c.lower() in enc.lower() for enc in campos_rag)])
if omitidos_rag:
    print(f"\n⚠️ Ojo: El RAG omitió la información sobre: {omitidos_rag}")
    print("Esto justifica la necesidad de avanzar hacia una arquitectura Map-Reduce para evitar pérdidas por mala recuperación vectorial.")
else:
    print("\n✅ El RAG recuperó todas las variables correctamente.")

--- GOLDEN DATASET (Variables Reales) ---
['Identificador del Crédito', 'Moneda del crédito', 'Tasa de Interés']

--- HALLAZGOS FULL CONTEXT ---
['identificador_credito', 'moneda_credito', 'tasa_interes']

--- HALLAZGOS RAG SIMPLE ---
['ATR_i^X', 'SP_i^X']

⚠️ Ojo: El RAG omitió la información sobre: {'Tasa de Interés', 'Identificador del Crédito', 'Moneda del crédito'}
Esto justifica la necesidad de avanzar hacia una arquitectura Map-Reduce para evitar pérdidas por mala recuperación vectorial.


## 5. Conclusión y Siguientes Pasos

1. **Full Context**: Tiende a ser más exhaustivo pero es computacionalmente caro y muy lento para producción masiva.
2. **RAG Simple**: Es muy rápido y económico, pero su éxito depende críticamente del Chunking y la Búsqueda Vectorial. Si la query no trae el "Chunk" donde está una variable, el modelo simplemente no la extraerá.
3. **Siguiente Paso (Map-Reduce)**: Implementar una iteración donde en lugar de consultar vectores, inyectemos todos los fragmentos asociados a un "Formulario" de forma fragmentada, analicemos uno por uno (Map) y unamos resultados al final (Reduce).